# CSC 2301 · Week 8 — NumPy arrays: calculations on a whole log

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/csc2301/notebooks/CSC2301_W8_numpy_well_log.ipynb)

**IUGB · BSc Mining / Petroleum Engineering** · Instructor: Laurent Kouadio

A modern well log has tens of thousands of samples. Loops work, but they are slow to write and slow to run.
**NumPy** stores numbers in **arrays** and applies a formula to every element at once.
The data are four logs from well KW-2 (synthetic), every 0.5 m from 2100 to 2400 m: gamma ray (GR),
bulk density (RHOB) and deep resistivity (RT).

**What you will practise:** creating arrays · element-wise arithmetic · `mean`, `max`, `sum` ·
Boolean masks · `&` and `|` · `np.clip`, `np.where` · indexing with a mask.

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/csc2301/'
FILES = [('data/well_log_kw2.csv', 'well_log_kw2.csv'), ('tests/w8/q1.py', 'tests/q1.py'), ('tests/w8/q2.py', 'tests/q2.py'), ('tests/w8/q3.py', 'tests/q3.py'), ('tests/w8/q4.py', 'tests/q4.py'), ('tests/w8/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

## 1. From a list to an array

In [ ]:
import numpy as np

values = [2.31, 2.45, 2.52, 2.28]
a = np.array(values)
print(a * 2)               # every element times 2
print(a - 1.0)
print(a.mean(), a.max(), a.shape, a.dtype)
print([v * 2 for v in values])   # the same with a list needs a loop

## 2. Loading the log

`np.loadtxt` reads a table of numbers into a 2-D array. Column `k` is `table[:, k]`.

In [ ]:
table = np.loadtxt("well_log_kw2.csv", delimiter=",", skiprows=1)
print(table.shape)          # (rows, columns)
depth = table[:, 0]
print(depth[:5], depth[-1])

## 3. Masks: choosing elements with a condition

A comparison on an array gives a **Boolean array**. Use it to count, or to select.

In [ ]:
x = np.array([45.0, 110.0, 38.0, 95.0, 52.0])
clean = x < 60
print(clean)                 # [ True False  True False  True]
print(clean.sum())           # True counts as 1 -> 3
print(x[clean])              # only the selected values
print(x[(x > 40) & (x < 100)])   # & = and, | = or; brackets are required
print(np.where(x < 60, "sand", "shale"))
print(np.clip(np.array([-0.2, 0.5, 1.3]), 0, 1))

---
## Exercises

From now on, **no `for` loop** over the samples: use array operations.

### Q1. The arrays

From `table`, create the arrays `gr` (column 1), `rhob` (column 2) and `rt` (column 3). Store the mean GR
in `mean_gr` and the highest density in `max_rhob`.

In [ ]:
gr = ...
rhob = ...
rt = ...
mean_gr = ...
max_rhob = ...
mean_gr, max_rhob

In [ ]:
grader.check("q1")

### Q2. Density porosity

For a sandstone (grain density 2.65 g/cm³) filled with water (1.0 g/cm³), porosity from the density log is

$$\phi_D = \frac{2.65 - \rho_b}{2.65 - 1.0}$$

Compute `phi_d` for every sample in one line.

In [ ]:
phi_d = ...
phi_d[:5]

In [ ]:
grader.check("q2")

### Q3. Sand samples

Create the mask `sand` (GR < 60), count the sand samples in `n_sand`, and store in `sand_phi_mean` the mean
density porosity of the sand samples only.

In [ ]:
sand = ...
n_sand = ...
sand_phi_mean = ...
n_sand, sand_phi_mean

In [ ]:
grader.check("q3")

### Q4. Shale volume, vectorised

In week 2 you wrote `vshale` with `if`. Now compute it for the whole log at once, with this well's clean
and shale lines: $V_{sh} = (GR - 25) / (130 - 25)$, clipped to 0–1 with `np.clip`. Store it in `vsh`.

In [ ]:
vsh = ...
vsh.min(), vsh.max()

In [ ]:
grader.check("q4")

### Q5. Net pay

A sample is **pay** when `vsh < 0.35` **and** `phi_d >= 0.10` **and** `rt >= 10` ohm·m (high resistivity:
hydrocarbons, not salt water). Build the mask `pay`, store the pay thickness in `net_pay_m`
(0.5 m per sample) and the depth of the shallowest pay sample in `top_pay_m`.

In [ ]:
pay = ...
net_pay_m = ...
top_pay_m = ...
net_pay_m, top_pay_m

In [ ]:
grader.check("q5")

### Going further (not checked)

* Time the two ways with `%timeit`: a Python loop computing `phi_d`, and the array formula.
* With `np.diff` on the pay mask (as integers), find where each pay zone starts and ends.

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()